# AnalystLab Africa ML Internship — Week 1-2
## Data Preprocessing & Exploratory Data Analysis
**Datasets:** Titanic (tabular) | IMDB Reviews (NLP)  
**Author:** ML Intern  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
import warnings, re, string
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titleweight'] = 'bold'
ACCENT = '#2563eb'
print("✅  Libraries loaded successfully.")

---
# PART 1 — Titanic Dataset
## 1.1 Load & Initial Inspection

In [ ]:
titanic = pd.read_csv('/mnt/user-data/uploads/Titanic-Dataset.csv')
print(f"Shape: {titanic.shape}")
titanic.head()

In [ ]:
titanic.info()

In [ ]:
titanic.describe(include='all').round(2)

## 1.2 Missing Value Analysis

In [ ]:
missing = titanic.isnull().sum()
missing_pct = (missing / len(titanic) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(miss_df)

fig, ax = plt.subplots(figsize=(7, 3))
miss_df['Missing %'].plot(kind='barh', ax=ax, color=ACCENT)
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Feature (Titanic)')
for i, v in enumerate(miss_df['Missing %']):
    ax.text(v + 0.3, i, f'{v}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/home/claude/titanic_missing.png', bbox_inches='tight')
plt.show()
print("\nCabin: 77% missing → drop. Age: 20% missing → median impute. Embarked: 2 missing → mode impute.")

## 1.3 Data Cleaning

In [ ]:
df = titanic.copy()

# 1. Drop Cabin (>77% missing) and non-informative columns
df.drop(columns=['Cabin', 'Ticket', 'Name', 'PassengerId'], inplace=True)

# 2. Impute Age with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# 3. Impute Embarked with mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

print("Missing values after cleaning:")
print(df.isnull().sum())
print(f"\nDataset shape: {df.shape}")

## 1.4 Outlier Detection (IQR Method)

In [ ]:
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']

fig, axes = plt.subplots(1, len(numeric_cols), figsize=(14, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color=ACCENT, width=0.5)
    ax.set_title(col)
plt.suptitle('Boxplots — Titanic Numeric Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/home/claude/titanic_outliers.png', bbox_inches='tight')
plt.show()

# IQR outlier counts
for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    print(f"{col}: {outliers} outliers detected")
print("\n→ Fare has extreme outliers (1st class). Capping applied below.")

In [ ]:
# Cap Fare at 99th percentile
cap = df['Fare'].quantile(0.99)
df['Fare'] = df['Fare'].clip(upper=cap)
print(f"Fare capped at 99th percentile: {cap:.2f}")

## 1.5 Encoding Categorical Variables

In [ ]:
# Binary encoding: Sex
df['Sex_enc'] = (df['Sex'] == 'male').astype(int)

# One-hot encoding: Embarked
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked', drop_first=True)
df = pd.concat([df, embarked_dummies], axis=1)

# Ordinal: Pclass already numeric (1=upper, 3=lower)

df.drop(columns=['Sex', 'Embarked'], inplace=True)
print("Encoded columns:", df.columns.tolist())
df.head(3)

## 1.6 Feature Scaling

In [ ]:
scaler = MinMaxScaler()
scale_cols = ['Age', 'Fare']
df[scale_cols] = scaler.fit_transform(df[scale_cols])

std_scaler = StandardScaler()
df_std = df.copy()
df_std[scale_cols] = std_scaler.fit_transform(df_std[scale_cols])

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for i, col in enumerate(scale_cols):
    df[col].hist(ax=axes[0][i], bins=25, color='steelblue', edgecolor='white')
    axes[0][i].set_title(f'{col} — MinMax Scaled')
    df_std[col].hist(ax=axes[1][i], bins=25, color='coral', edgecolor='white')
    axes[1][i].set_title(f'{col} — Standard Scaled')
plt.suptitle('Feature Scaling Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('/home/claude/titanic_scaling.png', bbox_inches='tight')
plt.show()

## 1.7 Exploratory Data Analysis

In [ ]:
# Survival rate overall
sr = titanic['Survived'].value_counts(normalize=True) * 100
print(f"Survived: {sr[1]:.1f}%  |  Not Survived: {sr[0]:.1f}%")

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(3, 3, figure=fig)

# 1. Survival count
ax1 = fig.add_subplot(gs[0, 0])
titanic['Survived'].value_counts().plot(kind='bar', ax=ax1, color=['#ef4444','#22c55e'], rot=0)
ax1.set_xticklabels(['Did Not Survive','Survived'])
ax1.set_title('Survival Count')
ax1.set_ylabel('Count')

# 2. Survival by Sex
ax2 = fig.add_subplot(gs[0, 1])
titanic.groupby('Sex')['Survived'].mean().mul(100).plot(kind='bar', ax=ax2, color=['#f97316','#3b82f6'], rot=0)
ax2.set_title('Survival Rate by Sex')
ax2.set_ylabel('Survival %')

# 3. Survival by Pclass
ax3 = fig.add_subplot(gs[0, 2])
titanic.groupby('Pclass')['Survived'].mean().mul(100).plot(kind='bar', ax=ax3, color=sns.color_palette('Blues_d', 3), rot=0)
ax3.set_title('Survival Rate by Pclass')
ax3.set_ylabel('Survival %')

# 4. Age distribution
ax4 = fig.add_subplot(gs[1, 0:2])
sns.histplot(data=titanic.dropna(subset=['Age']), x='Age', hue='Survived', bins=30, ax=ax4, palette={0:'#ef4444', 1:'#22c55e'})
ax4.set_title('Age Distribution by Survival')

# 5. Fare distribution
ax5 = fig.add_subplot(gs[1, 2])
sns.boxplot(data=titanic, x='Survived', y='Fare', ax=ax5, palette={0:'#ef4444', 1:'#22c55e'})
ax5.set_title('Fare by Survival')
ax5.set_xticklabels(['No','Yes'])

# 6. Pclass × Sex heatmap
ax6 = fig.add_subplot(gs[2, 0:2])
heat = titanic.groupby(['Pclass','Sex'])['Survived'].mean().unstack().mul(100).round(1)
sns.heatmap(heat, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax6, linewidths=0.5)
ax6.set_title('Survival Rate (%) — Pclass × Sex')

# 7. Correlation heatmap
ax7 = fig.add_subplot(gs[2, 2])
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', ax=ax7, annot=False, linewidths=0.3)
ax7.set_title('Feature Correlation')

plt.suptitle('Titanic — Exploratory Data Analysis', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/home/claude/titanic_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation with target
print("Correlation with Survived (cleaned df):")
print(df.corr()['Survived'].sort_values(ascending=False).round(3))

---
# PART 2 — IMDB Dataset
## 2.1 Load & Initial Inspection

In [ ]:
imdb = pd.read_csv('/mnt/user-data/uploads/IMDB_Dataset.csv')
print(f"Shape: {imdb.shape}")
print("\nSentiment distribution:")
print(imdb['sentiment'].value_counts())
imdb.head(2)

## 2.2 Text Preprocessing

In [ ]:
def clean_text(text):
    text = text.lower()                                         # lowercase
    text = re.sub(r'<.*?>', ' ', text)                          # remove HTML tags
    text = re.sub(r'http\S+|www\.\S+', '', text)             # remove URLs
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\d+', '', text)                            # remove digits
    text = re.sub(r'\s+', ' ', text).strip()                   # normalise whitespace
    return text

imdb['review_clean'] = imdb['review'].apply(clean_text)
imdb['review_length_raw']   = imdb['review'].apply(len)
imdb['review_length_clean'] = imdb['review_clean'].apply(len)
imdb['word_count'] = imdb['review_clean'].apply(lambda x: len(x.split()))

print("Sample cleaned review:")
print(imdb['review_clean'].iloc[0][:300])

## 2.3 Encoding Sentiment Label

In [ ]:
imdb['label'] = (imdb['sentiment'] == 'positive').astype(int)
print("Label distribution:", dict(imdb['label'].value_counts()))

## 2.4 Exploratory Data Analysis — IMDB

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Sentiment distribution
imdb['sentiment'].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['#22c55e','#ef4444'], rot=0)
axes[0,0].set_title('Sentiment Distribution')
axes[0,0].set_ylabel('Count')

# 2. Review length by sentiment
sns.boxplot(data=imdb, x='sentiment', y='review_length_raw', ax=axes[0,1],
    palette={'positive':'#22c55e','negative':'#ef4444'})
axes[0,1].set_title('Raw Review Length by Sentiment')
axes[0,1].set_ylabel('Characters')

# 3. Word count distribution
sns.histplot(data=imdb, x='word_count', hue='sentiment', bins=40,
    ax=axes[1,0], palette={'positive':'#22c55e','negative':'#ef4444'}, alpha=0.7)
axes[1,0].set_title('Word Count Distribution')
axes[1,0].set_xlim(0, 1200)

# 4. Average word count per sentiment
imdb.groupby('sentiment')['word_count'].mean().plot(kind='bar', ax=axes[1,1],
    color=['#ef4444','#22c55e'], rot=0)
axes[1,1].set_title('Average Word Count by Sentiment')
axes[1,1].set_ylabel('Avg Words')

plt.suptitle('IMDB Reviews — EDA', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/imdb_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top words per sentiment
from collections import Counter

def top_words(series, n=15):
    words = ' '.join(series).split()
    # remove very common stop words
    stopwords = {'the','a','and','of','to','is','in','it','this','that','was',
                 'for','as','with','his','her','he','she','they','are','an','be',
                 'have','had','but','on','at','by','not','from','or','i','its',
                 'were','so','all','we','one','has','my','there','what','if','would',
                 'no','their','been','more','about','up','out','do','when','him',
                 'me','can','just','into','your','like','very'}
    words = [w for w in words if w not in stopwords and len(w) > 2]
    return Counter(words).most_common(n)

pos_words = top_words(imdb[imdb['sentiment']=='positive']['review_clean'])
neg_words = top_words(imdb[imdb['sentiment']=='negative']['review_clean'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, words, title, color in [
    (axes[0], pos_words, 'Top 15 Words — Positive', '#22c55e'),
    (axes[1], neg_words, 'Top 15 Words — Negative', '#ef4444')]:
    wds, cnts = zip(*words)
    ax.barh(wds[::-1], cnts[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frequency')
plt.suptitle('Most Common Words by Sentiment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/imdb_words.png', bbox_inches='tight')
plt.show()

---
## Summary of Findings

### Titanic Dataset
| Observation | Detail |
|---|---|
| **Missing Data** | Cabin (77%) dropped; Age (20%) median-imputed; Embarked (0.2%) mode-imputed |
| **Outliers** | Fare has extreme values — capped at 99th percentile |
| **Survival Rate** | ~38% overall; Women survived at ~74% vs men at ~19% |
| **Class Effect** | 1st class survival ~63%, 2nd ~47%, 3rd ~24% |
| **Age** | Children (<10) had higher survival rates; middle-aged men fared worst |
| **Encoding** | Sex → binary; Embarked → one-hot; Pclass already ordinal numeric |
| **Scaling** | Age & Fare normalised with MinMax (range [0,1]) |

### IMDB Dataset
| Observation | Detail |
|---|---|
| **Balance** | Perfectly balanced — 25,000 positive, 25,000 negative |
| **Text Noise** | HTML tags, URLs, punctuation, digits removed |
| **Review Length** | Positive reviews tend to be slightly longer on average |
| **Vocabulary** | "film", "movie", "great", "good" dominate positives; "bad", "worst", "waste" dominate negatives |
| **Label Encoding** | positive → 1, negative → 0 |

### Next Steps
- Titanic → Feature engineering (Title from Name, FamilySize = SibSp+Parch+1), then modelling (Logistic Regression, Random Forest)
- IMDB → TF-IDF or word embeddings (Word2Vec/GloVe), then text classification (Logistic Regression, LSTM)

In [ ]:
df.to_csv('/home/claude/titanic_cleaned.csv', index=False)
imdb[['review_clean','sentiment','label','word_count']].to_csv('/home/claude/imdb_cleaned.csv', index=False)
print("✅  Cleaned datasets saved.")
print("    titanic_cleaned.csv:", df.shape)
print("    imdb_cleaned.csv:   ", imdb.shape)